<a href="https://colab.research.google.com/github/Nurdaylight/A-Karpathy-repl/blob/main/Translation_Mapping_14_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import re
from functools import lru_cache
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
!wget https://raw.githubusercontent.com/Nurdaylight/Study/main/en_CH1.txt -O ru_test.txt
import string

with open('ru_test.txt', 'r', encoding='utf-8') as f:
    text = f.read()


--2026-02-16 14:58:18--  https://raw.githubusercontent.com/Nurdaylight/Study/main/en_CH1.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7937 (7.8K) [text/plain]
Saving to: ‘ru_test.txt’

ru_test.txt         100%[===================>]   7.75K  --.-KB/s    in 0.001s  

2026-02-16 14:58:18 (10.7 MB/s) - ‘ru_test.txt’ saved [7937/7937]



In [3]:
text= text.replace("\r\n", " ")
text= text.replace("\n", " ")
text

'  Book I. The History Of A Family     Chapter I. Fyodor Pavlovitch Karamazov   Alexey Fyodorovitch Karamazov was the third son of Fyodor Pavlovitch Karamazov, a land owner well known in our district in his own day, and still remembered among us owing to his gloomy and tragic death, which happened thirteen years ago, and which I shall describe in its proper place. For the present I will only say that this “landowner”—for so we used to call him, although he hardly spent a day of his life on his own estate—was a strange type, yet one pretty frequently to be met with, a type abject and vicious and at the same time senseless. But he was one of those senseless persons who are very well capable of looking after their worldly affairs, and, apparently, after nothing else. Fyodor Pavlovitch, for instance, began with next to nothing; his estate was of the smallest; he ran to dine at other men’s tables, and fastened on them as a toady, yet at his death it appeared that he had a hundred thousand r

In [4]:
etor_translate=text[159:265]
etor_translate

'a land owner well known in our district in his own day, and still remembered among us owing to his gloomy '

In [5]:


key_list = ["земля", "владелец"]
messages = [
    {
        "role": "user",
        "content": (
            f"Translate only key words in the list {key_list} in the following segment into Russian, do not change any other word. Without additional explanation. The text is from Brothers Karamazov "
            "without additional explanation.\n\n"
            f"{etor_translate}"
        )
    },
]



In [6]:
### По минимуму просто грузим базу модели
tokenizer = AutoTokenizer.from_pretrained("tencent/HY-MT1.5-1.8B")
model = AutoModelForCausalLM.from_pretrained("tencent/HY-MT1.5-1.8B").to("cuda")

Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'beta_slow', 'mscale_all_dim', 'alpha', 'beta_fast', 'rope_theta', 'mscale'}
Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'beta_slow', 'mscale_all_dim', 'alpha', 'beta_fast', 'rope_theta', 'mscale'}


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [7]:

#MODEL_NAME = "tencent/HY-MT1.5-1.8B"

# ---- load model (silence tied-weights warning) ----
#config = AutoConfig.from_pretrained(MODEL_NAME)
#onfig.tie_word_embeddings = False

#tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#model = AutoModelForCausalLM.from_pretrained(
 #   MODEL_NAME,
 #   config=config,
#  device_map="auto",
 #   torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
#)
#model.eval()

In [8]:

### Шаманим с применением модели

WORD_RE = re.compile(r"\b[A-Za-z][A-Za-z'-]*\b")


def _prompt_for(word: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": f"Translate into Russian:\n{word}"}],
        tokenize=False,
        add_generation_prompt=False,
    )

### Надо проверить какие кандидаты выдает полностью.
### So far only fisrt element what if I want more of the chunk?
def _normalize_ru_candidate(s: str) -> str:
    # Take first "word-like" chunk and strip punctuation/spaces.
    s = s.strip()
    s = re.sub(r"^[\s\"'“”‘’\(\)\[\]\{\}\-–—,.:;!?]+", "", s)
    s = re.sub(r"[\s\"'“”‘’\(\)\[\]\{\}\-–—,.:;!?]+$", "", s)
    # If model outputs multiple words, keep the first token group.
    s = s.split()[0] if s else ""
    return s

### dedupe loop can be shorter using a dict.
@torch.inference_mode()
def _topk_translations(word: str, k: int = 3, num_beams: int = 6) -> list[str]:
    """
    Get top-k decoded translations from beam search.
    These are the 'before adjustment' predictions.
    """
    prompt = _prompt_for(word)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outs = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        num_beams=num_beams,
        num_return_sequences=k,
        early_stopping=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    prompt_len = inputs.input_ids.shape[1]
    preds = []
    for seq in outs:
        gen = tokenizer.decode(seq[prompt_len:], skip_special_tokens=True)
        gen = _normalize_ru_candidate(gen)
        if gen:
            preds.append(gen)

    # Deduplicate while preserving order
    seen = set()
    uniq = []
    for p in preds:
        key = p.casefold()
        if key not in seen:
            seen.add(key)
            uniq.append(p)
    return uniq[:k]

### This is high cost. I dont really need to call softmax on all.
### I can do inference on logits directly here.
@torch.inference_mode()
def _loglik(prompt_text: str, continuation_text: str) -> float:
    """
    Total log-likelihood of emitting continuation_text after prompt_text.
    """
    p = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    c = tokenizer(continuation_text, add_special_tokens=False, return_tensors="pt").to(model.device)

    input_ids = torch.cat([p.input_ids, c.input_ids], dim=1)
    labels = input_ids.clone()
    labels[:, : p.input_ids.shape[1]] = -100

    loss = model(input_ids=input_ids, labels=labels).loss
    return -loss.item() * c.input_ids.shape[1]

### Порог (threshold) поменять на квантили
@lru_cache(maxsize=4096)
def _choose_replacement(word_lower: str, ru_dict_tuple: tuple[str, ...], threshold: float) -> str | None:
    """
    1) Get top-3 translations (before adjustment)
    2) Only consider RU dictionary entries that appear in that top-3 list
    3) If any match, apply adjustment: P(ru) vs P(keep word) must exceed threshold
    """
    top3 = _topk_translations(word_lower, k=3)
    top3_set = {t.casefold() for t in top3}

    # Candidates allowed ONLY if in top-3 predictions

    ### Надо делать не на каждое слово, пока хз как.
    ### создать зарание mappings для лучших только?
    allowed = [ru for ru in ru_dict_tuple if ru.casefold() in top3_set]
    if not allowed:
        return None

    prompt = _prompt_for(word_lower)

    # Among allowed, pick the one with best adjusted probability vs keep
    best_ru = None
    best_prob = -1.0

    ll_keep = _loglik(prompt, word_lower)

    for ru in allowed:
        ll_ru = _loglik(prompt, ru)

        # prob_ru in a 2-way softmax: ru vs keep
        prob_ru = torch.softmax(torch.tensor([ll_ru, ll_keep], dtype=torch.float64), dim=0)[0].item()

        if prob_ru > best_prob:
            best_prob = prob_ru
            best_ru = ru

    return best_ru if best_prob >= threshold else None


def translate_keywords_in_text(text: str, ru_dictionary: list[str], threshold: float = 0.75) -> str:
    ru_dict_t = tuple(ru_dictionary)

    def repl(m: re.Match) -> str:
        w = m.group(0)
        ru = _choose_replacement(w.lower(), ru_dict_t, threshold)
        if ru is None:
            return w
        # Basic capitalization rule
        return ru.capitalize() if w[0].isupper() else ru

    return WORD_RE.sub(repl, text)


In [9]:


# ---- example ----
dictionary = ["жизнь"]  # only replace if "жизнь" is in top-3 raw predictions
print(translate_keywords_in_text("Life is short", dictionary, threshold=0.75))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Жизнь is short


In [10]:
!wget https://raw.githubusercontent.com/hingston/russian/master/100000-russian-words.txt -O words.txt

with open('words.txt', 'r', encoding='utf-8') as f:
    words = f.read()
words = words.split('\n')
print(words[:10])

--2026-02-16 14:58:25--  https://raw.githubusercontent.com/hingston/russian/master/100000-russian-words.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1671085 (1.6M) [text/plain]
Saving to: ‘words.txt’

words.txt           100%[===================>]   1.59M  --.-KB/s    in 0.06s   

2026-02-16 14:58:25 (28.5 MB/s) - ‘words.txt’ saved [1671085/1671085]

['и', 'в', 'не', 'на', 'я', 'что', 'быть', 'quot', 'с', 'он']


In [11]:
# ---------- example ----------
dictionary = words[100:250] #["земля","землевладелец","его"]

print(translate_keywords_in_text(text[159:259], dictionary))

a земля owner хорошо known in our district in his own day, and still remembered между us owing to his g
